# 第 4 章 §4.3（LangChain 版）

与书里一致，仅标 **【适配】** 处有改动。kernel 选 `Python (agentic-book)`。

中间结果由 `show()` 打印：它把上游 dict 原样透传，不改变链的结构。

In [1]:
import sys
import asyncio
import threading
from pathlib import Path

# 【适配】找到含 common/ 的根目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from common import make_llm

# --- Configuration ---
try:
    llm = make_llm(temperature=0.7)                                # 【适配】原 ChatOpenAI(model="gpt-4o-mini")
    print(f"Language model initialized: {getattr(llm, 'model_name', '?')}")
except Exception as e:
    print(f"Error initializing language model: {e}", file=sys.stderr)
    raise RuntimeError("模型初始化失败")                            # 【适配】原 sys.exit(1) 会杀掉 kernel


# --- Define Chain Components ---
generation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a short, simple product description for a new smart coffee mug."),
        ("user", "{product_details}")
    ])
    | llm
    | StrOutputParser()
)

critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Critique the following product description based on clarity, conciseness, and appeal.
        Provide specific suggestions for improvement."""),
        ("user", "Product Description to Critique:\n{initial_description}")
    ])
    | llm
    | StrOutputParser()
)

refinement_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Based on the original product details and the following critique,
        rewrite the product description to be more effective.

        Original Product Details: {product_details}
        Critique: {critique}

        Refined Product Description:"""),
        ("user", "")
    ])
    | llm
    | StrOutputParser()
)


# --- 打印中间结果（原书没有；透传 dict，不改链结构）---
def show(key: str):
    def _f(x):
        print(f"\n--- {key} ---")
        print(x[key])
        return x
    return RunnableLambda(_f)


# --- Build the Full Reflection Chain (Refactored) ---
full_reflection_chain = (
    RunnablePassthrough.assign(
        initial_description=generation_chain
    )
    | show("initial_description")
    | RunnablePassthrough.assign(
        critique=critique_chain
    )
    | show("critique")
    | refinement_chain
)


# --- Run the Chain ---
async def run_reflection_example(product_details: str):
    """Runs the LangChain reflection example with product details."""
    print(f"\n--- Running Reflection Example for Product: '{product_details}' ---")
    try:
        final_refined_description = await full_reflection_chain.ainvoke(
            {"product_details": product_details}
        )
        print("\n--- Final Refined Product Description ---")
        print(final_refined_description)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")


def run_in_thread(coro_fn):
    """【适配】notebook 内已有事件循环，asyncio.run 会报错 → 在独立线程里跑。"""
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(coro_fn())
        except BaseException as e:            # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


if __name__ == "__main__":
    test_product_details = "A mug that keeps coffee hot and can be controlled by a smartphone app."
    run_in_thread(lambda: run_reflection_example(test_product_details))

Language model initialized: deepseek-flash

--- Running Reflection Example for Product: 'A mug that keeps coffee hot and can be controlled by a smartphone app.' ---

--- initial_description ---
**Smart Coffee Mug**

Keep your coffee hot from the first sip to the last. This smart mug connects to a smartphone app, so you can easily set and adjust the temperature to keep every cup just right.

--- critique ---
## Overall critique

The description is clear enough to understand the basic product, and it is concise. However, it is generic and undersells a “smart” product: it gives no proof points, no differentiation, and a few phrases are vague or redundant. Appeal is mild—it states a benefit but doesn’t create desire or answer common buyer questions.

## Clarity

**Strengths**
- The core benefit is immediately understandable: temperature-controlled coffee.
- “Connects to a smartphone app” quickly signals the “smart” feature.

**Issues**
- “Keep your coffee hot” conflicts slightly with “set 